# Bibliotecas

In [2]:
import os
import sys
import random
import time
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Gymnasium e wrappers
import gymnasium as gym

# Stable-Baselines3
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecMonitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, ProgressBarCallback

# Wrappers para customizações do LunarLander-v3

In [3]:
class RewardShapingWrapper_OLD(gym.Wrapper):
    """
    Reward shaping baseado nas observações do LunarLander.
    Obs: [x, y, vx, vy, angle, angular_vel, leg1_contact, leg2_contact]
    """
    def __init__(self, env, angle_penalty=20.0, dist_penalty=0.5, height_penalty=2.0, land_bonus=1000.0, time_penalty=5e-5, below_penalty=0.2):
        super().__init__(env)
        self.time = time.time()
        self.angle_penalty = angle_penalty
        self.dist_penalty = dist_penalty
        self.land_bonus = land_bonus
        self.height_penalty = height_penalty
        self.time_penalty = time_penalty
        self.below_penalty = below_penalty
        print(f"[RewardShapingWrapper] Exponential penalties: height_penalty={height_penalty}, dist_penalty={dist_penalty}, time_penalty={time_penalty}")

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        # Calcular reward shaping baseado na observação
        shaping = 0.0
        try:
            # Penalizar por quantidade de steps (incentivar aterragem rápida)
            time_elapsed = time.time() - self.time
            shaping -= self.time_penalty * time_elapsed


            # LunarLander obs: [x, y, vx, vy, angle, angular_vel, leg1_contact, leg2_contact]
            x = float(obs[0])
            y = float(obs[1])
            vx = float(obs[2])
            vy = float(obs[3])
            angle = float(obs[4])
            angular_vel = float(obs[5])
            leg1_contact = bool(obs[6])
            leg2_contact = bool(obs[7])
            helipad_x = getattr(self.env.unwrapped, "helipad_x", 0.0)
            helipad_y = getattr(self.env.unwrapped, "helipad_y", 0.0)

            # PENALIZAR: Ângulo e velocidade angular (queremos horizontal e estável)
            shaping -= self.angle_penalty * (abs(angle) + abs(angular_vel))

            # PENALIZAR: Distância horizontal ao centro (x=0 é o helipad)
            dist_x = helipad_x - x
            x_penalty = self.dist_penalty * (np.exp(abs(dist_x)) - 1)
            shaping -= x_penalty
            
            # PENALIZAR: Altura (y > 0 = acima do solo)
            # Penalização EXPONENCIAL: quanto mais alto, muito pior
            height_above = max(0.0, y - helipad_y)
            y_penalty = self.height_penalty * (np.exp(height_above) - 1)

            # Penalizar se estiver num ponto mais baixo do que o helipad
            if height_above < -0.1:
                shaping -= self.below_penalty * y_penalty

            # Penalalizar velocidade consoante mais perto do solo
            if height_above < 1:
                vel_penalty = abs(vy) * (2 - height_above)  # Quanto mais perto do solo, maior a penalização
                shaping -= vel_penalty * 2.0
                shaping -= y_penalty
            # Penalizar ainda mais por altura superior a 0.6 em relação ao helipad
            else:
                shaping -= y_penalty ** 2

            # Penalizar se estivar a ganhar altitude
            if vy > 0 and height_above > 0:
                shaping -= vy * 0.5

            # Penalizar se pousar longe do helipad
            if leg1_contact or leg2_contact:
                shaping -= x_penalty * 1.0

            # Pequeno bónus por tocar com ambas as pernas
            if leg1_contact and leg2_contact and vy < 0.2 and not terminated:
                print(f"[RewardShapingWrapper] Landed! Adding bonus of {self.land_bonus}")
                shaping += self.land_bonus        

            

        except Exception as e:
            print(f"[RewardShaping ERROR] {e}")
            import traceback
            traceback.print_exc()
            shaping = 0.0
        
        # Adicionar shaping ao reward original
        shaped_reward = reward + shaping
        
        return obs, shaped_reward, terminated, truncated, info

In [4]:
class RewardShapingWrapper(gym.Wrapper):
    """Reward shaping SIMPLIFICADO para LunarLander"""
    def __init__(self, env, shaping_scale=0.1):
        super().__init__(env)
        self.shaping_scale = shaping_scale
        self.step_count = 0
        print(f"[RewardShapingWrapper] Using simplified shaping (scale={shaping_scale})")

    def reset(self, **kwargs):
        self.step_count = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.step_count += 1
        
        shaping = 0.0
        try:
            x, y, vx, vy, angle, angular_vel, leg1, leg2 = obs[:8]

            # 1. Penalizar distância horizontal (LINEAR)
            shaping -= abs(x) * 0.3

            # 2. Penalizar altura (LINEAR, não exponencial)
            shaping -= max(0, y) * 0.5
            
            # 3. Penalizar ângulo e rotação
            shaping -= (abs(angle) + abs(angular_vel)) * 0.5
            
            # 4. Recompensar descida CONTROLADA perto do solo
            if y < 0.5 and vy < 0:  # A descer perto do solo
                # Vy ideal: -0.3 a -0.5
                vel_quality = 1.0 - abs(vy + 0.4) / 0.4
                shaping += max(0, vel_quality) * 0.3
            
            # 5. Bonus moderado por aterragem
            if leg1 and leg2 and abs(x) < 0.2:
                shaping += 10.0  # REDUZIDO de 1000

            # 6. Penalizar por tempo (muito leve)
            shaping -= self.step_count * 1e-4
            
            # Escalar shaping
            shaping *= self.shaping_scale
            
        except Exception as e:
            print(f"[RewardShaping ERROR] {e}")
            shaping = 0.0
        
        return obs, reward + shaping, terminated, truncated, info

In [5]:

class WindForceWrapper(gym.Wrapper):
    """Aplica perturbação lateral (vento) à velocidade horizontal 'observada'."""
    def __init__(self, env, wind_strength=0.02, deterministic=False):
        super().__init__(env)
        self.wind_strength = wind_strength
        self.deterministic = deterministic

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done_flag = (terminated or truncated)

        if isinstance(obs, np.ndarray) and obs.shape[0] >= 3:
            wind = self.wind_strength if self.deterministic else self.wind_strength * (2*np.random.rand()-1)
            obs = obs.copy()
            obs[2] += wind
            reward -= abs(wind) * 5.0

        # devolver no mesmo formato do original
        return obs, reward, terminated, truncated, info

In [6]:

class ObservationNoiseWrapper(gym.ObservationWrapper):
    """Add gaussian noise to observations to make the task more robust.
    """
    def __init__(self, env, noise_std=0.01):
        super().__init__(env)
        self.noise_std = noise_std


    def observation(self, observation):
        if isinstance(observation, np.ndarray):
            return observation + np.random.normal(scale=self.noise_std, size=observation.shape)
        return observation

In [7]:

class ActionSmoothingWrapper(gym.ActionWrapper):
    """
    Suaviza as ações usando média móvel exponencial (EMA) para reduzir oscilações.
    Útil para estabilizar o controle durante o pouso.
    """
    def __init__(self, env, smoothing_factor=0.3):
        super().__init__(env)
        self.smoothing_factor = smoothing_factor  # alpha para EMA (0-1)
        self.previous_action = None
        print(f"[ActionSmoothingWrapper] smoothing_factor={smoothing_factor}")

    def reset(self, **kwargs):
        self.previous_action = None
        return self.env.reset(**kwargs)

    def action(self, action):
        """
        Aplica suavização exponencial: 
        smoothed_action = alpha * current_action + (1-alpha) * previous_action
        """
        if self.previous_action is None:
            # Primeira ação do episódio, sem suavização
            self.previous_action = action
            return action
        
        # Aplicar EMA
        smoothed_action = (self.smoothing_factor * action + 
                          (1 - self.smoothing_factor) * self.previous_action)
        
        # Arredondar para inteiro (ações discretas: 0, 1, 2, 3)
        smoothed_action = int(np.round(np.clip(smoothed_action, 0, 3)))
        
        self.previous_action = smoothed_action
        return smoothed_action


# Criação do ambiente customizado

In [8]:
def make_env_factory(env_id='LunarLander-v3', seed=None, config_name='orig', monitor_dir=None):
    """
    Retorna uma função _init compatível com DummyVecEnv que:
     - cria env,
     - aplica wrappers consoante config_name,
     - envolve com Monitor(escreve monitor.csv em monitor_dir).
    config_name em {'orig', 'reward', 'wind', 'noise', 'all'}
    """
    def _init():
        env = gym.make(env_id)
        if seed is not None:
            # seed reset (Gymnasium)
            try:
                env.reset(seed=seed)
            except TypeError:
                env.reset()
            env.action_space.seed(seed)
            env.observation_space.seed(seed)

        # Aplicar wrappers conforme config
        if 'reward' in config_name:
            print("[MAKING ENV] Applying RewardShapingWrapper")
            env = RewardShapingWrapper(env)
        if 'wind' in config_name:
            print("[MAKING ENV] Applying WindForceWrapper")
            env = WindForceWrapper(env, wind_strength=0.01, deterministic=False)
        if 'noise' in config_name:
            print("[MAKING  ENV] Applying ObservationNoiseWrapper")
            env = ObservationNoiseWrapper(env, noise_std=0.02)
        if 'smooth' in config_name:
            print("[MAKING ENV] Applying ActionSmoothingWrapper")
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        if config_name == 'all':
            print("[MAKING ENV] Applying ALL wrappers")
            env = RewardShapingWrapper(env)
            env = WindForceWrapper(env, wind_strength=0.03, deterministic=False)
            env = ObservationNoiseWrapper(env, noise_std=0.02)
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        # caso 'orig' => nenhum wrapper

        # Monitor: grava os episódios neste ficheiro
        if monitor_dir is not None:
            os.makedirs(monitor_dir, exist_ok=True)
            monitor_file = os.path.join(monitor_dir, 'monitor.csv')
            env = Monitor(env, filename=monitor_file)
        else:
            env = Monitor(env)



        return env    
    return _init

In [9]:
def make_env_factory_multiple_configs(env_id='LunarLander-v3', seed=None, config_name='orig', monitor_dir=None):
    """
    Retorna uma função _init compatível com DummyVecEnv que:
     - cria env,
     - aplica wrappers consoante config_name,
     - envolve com Monitor(escreve monitor.csv em monitor_dir).
    config_name em {'orig', 'reward', 'wind', 'noise', 'all'}
    """
    def _init():
        env = gym.make(env_id)
        if seed is not None:
            # seed reset (Gymnasium)
            try:
                env.reset(seed=seed)
            except TypeError:
                env.reset()
            env.action_space.seed(seed)
            env.observation_space.seed(seed)

        # Aplicar wrappers conforme config
        if 'reward' in config_name:
            print("[CREATING ENV] Applying RewardShapingWrapper")
            env = RewardShapingWrapper(env)
        if 'wind' in config_name:
            print("[CREATING ENV] Applying WindForceWrapper")
            env = WindForceWrapper(env, wind_strength=0.01, deterministic=False)
        if 'noise' in config_name:
            print("[CREATING ENV] Applying ObservationNoiseWrapper")
            env = ObservationNoiseWrapper(env, noise_std=0.02)
        if 'smooth' in config_name:
            print("[CREATING ENV] Applying ActionSmoothingWrapper")
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        if config_name == 'all':
            print("[CREATING ENV] Applying ALL wrappers")
            env = RewardShapingWrapper(env)
            env = WindForceWrapper(env, wind_strength=0.03, deterministic=False)
            env = ObservationNoiseWrapper(env, noise_std=0.02)
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        
        # caso 'orig' => nenhum wrapper
        
        # Monitor: grava os episódios neste ficheiro
        if monitor_dir is not None:
            os.makedirs(monitor_dir, exist_ok=True)
            monitor_file = os.path.join(monitor_dir, 'monitor.csv')
            env = Monitor(env, filename=monitor_file)
        else:
            env = Monitor(env)



        return env    
    return _init

# Função de treino do PPO

In [10]:
def train_configs(config_name='orig', seed=0, timesteps=300_000, hyperparams=None, out_dir='./experiments', name='experiment', n_envs=4):
    """
    Treina um PPO para a configuração especificada.
    - Guarda modelo e monitor.csv em out_root/config_name/seedX/
    - hyperparams: dict que sobrepõe os defaults do PPO (learning_rate,n_steps,batch_size,n_epochs,...)
    - n_envs: número de ambientes paralelos (default=4, usar 1 para single env)
    """
    if hyperparams is None:
        hyperparams = {}

    # GPU auto-detect
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\n{'='*60}")
    print(f"TRAINING CONFIGURATION")
    print(f"{'='*60}")
    print(f"  Config: {config_name}")
    print(f"  Seed: {seed}")
    print(f"  Timesteps: {timesteps:,}")
    print(f"  Device: {device.upper()}")
    print(f"  Parallel Envs: {n_envs}")
    print(f"  Hyperparams: {hyperparams if hyperparams else 'defaults'}")
    print(f"  Output: {out_dir}")
    print(f"{'='*60}\n")

    # reproducibilidade
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    # criar env factory (monitor dentro da pasta out_dir)
    vec_env = None
    eval_env = None
    
    try:
        # Paralelização: usar SubprocVecEnv se n_envs > 1, senão DummyVecEnv
        if n_envs > 1:
            env_fns = [make_env_factory(seed=seed+i, config_name=config_name, monitor_dir=out_dir) 
                       for i in range(n_envs)]
            vec_env = SubprocVecEnv(env_fns)
        else:
            env_fn = make_env_factory(seed=seed, config_name=config_name, monitor_dir=out_dir)
            vec_env = DummyVecEnv([env_fn])
        
        vec_env = VecMonitor(vec_env)

        # policy defaults
        policy_kwargs = dict(activation_fn=torch.nn.Tanh, net_arch=[dict(pi=[256,256], vf=[256,256])])

        # PPO defaults (podes sobrepor via hyperparams)
        ppo_defaults = dict(
            policy='MlpPolicy',
            env=vec_env,
            verbose=1,
            seed=seed,
            learning_rate=1e-4,
            n_steps=2048,
            batch_size=64,
            n_epochs=10,
            gamma=0.99,
            gae_lambda=0.95,
            ent_coef=0.01,
            vf_coef=0.5,
            clip_range=0.2,
            policy_kwargs=policy_kwargs,
            tensorboard_log=os.path.join(out_dir, 'tb'),
            device=device  # GPU auto-detect
        )
        # update defaults with provided hyperparams
        ppo_defaults.update(hyperparams)

        model = PPO(**ppo_defaults)

        # callbacks: evaluation, checkpoints e progress bar
        eval_env = DummyVecEnv([make_env_factory(seed=seed+1000, config_name=config_name, monitor_dir=None)])
        eval_env = VecMonitor(eval_env)
        eval_callback = EvalCallback(eval_env, best_model_save_path=out_dir,
                                     log_path=out_dir, eval_freq=max(10_000, ppo_defaults['n_steps']*2),
                                     n_eval_episodes=5, deterministic=True, render=False)
        checkpoint_callback = CheckpointCallback(save_freq=max(50_000, ppo_defaults['n_steps']*5),
                                                 save_path=out_dir, name_prefix='ppo_checkpoint')
        progress_callback = ProgressBarCallback()

        # Treinar com progress bar
        model.learn(total_timesteps=timesteps, 
                   callback=[eval_callback, checkpoint_callback, progress_callback],
                   progress_bar=True)

        model_path = os.path.join(out_dir, f'ppo_{name}_seed{seed}.zip')
        model.save(model_path)
        print(f"\n[TRAIN] ✅ Saved model: {model_path}")
        
        return model_path, out_dir
        
    finally:
        # Garantir que ambientes fecham mesmo com erros
        if vec_env is not None:
            vec_env.close()
        if eval_env is not None:
            eval_env.close()

# Função de avaliação com critério revisado

In [11]:
def evaluate_custom(model_path, config_name='orig', seed=None, episodes=50):
    """
    Avalia o modelo (usa DummyVecEnv com mesma config). Critério de sucesso:
     - em qualquer step do episódio both legs touched, OR total_reward >= 200
    Retorna dicionário com métricas e lista de recompensas por episódio.
    """
    vec_env = None
    
    try:
        # carregar env (para avaliação, monitor não é necessário)
        env_fn = make_env_factory(seed=seed, config_name=config_name, monitor_dir=None)
        vec_env = DummyVecEnv([env_fn])
        vec_env = VecMonitor(vec_env)

        model = PPO.load(model_path, env=vec_env)

        # avaliação rápida via evaluate_policy (apenas para ter mean/std)
        mean_reward, std_reward = evaluate_policy(model, vec_env, n_eval_episodes=min(10, episodes), deterministic=True)
        print(f"[EVAL] quick evaluate_policy: mean={mean_reward:.2f} std={std_reward:.2f}")

        # per-episode sampling para success metric
        successes = 0
        crashes = 0
        rewards = []
        episode_lengths = []
        
        for _ in tqdm(range(episodes), desc="Evaluating episodes", unit="ep"):
            reset_out = vec_env.reset()
            obs = reset_out[0] if isinstance(reset_out, tuple) else reset_out

            done = False
            total_r = 0.0
            steps = 0
            landed_flag = False
            
            while not done:
                action, _ = model.predict(obs, deterministic=True)
                step_out = vec_env.step(action)
                # VecEnv.step returns (obs, reward, done, info)
                obs, reward, done, info = step_out
                steps += 1
                
                # reward pode ser array shape (1,), garantir float
                try:
                    total_r += float(np.array(reward).item())
                except Exception:
                    total_r += float(reward)

                # verificar contacto das pernas na observação atual
                try:
                    last_obs = obs[0]  # porque DummyVecEnv usa batch dimension
                    if bool(last_obs[6]) and bool(last_obs[7]):
                        landed_flag = True
                except Exception:
                    pass

            if landed_flag or total_r >= 200:
                successes += 1
            elif total_r < -100:  # Episódio terminou mal (crash)
                crashes += 1
            
            rewards.append(total_r)
            episode_lengths.append(steps)

        rewards = np.array(rewards)
        episode_lengths = np.array(episode_lengths)
        
        result = {
            'mean_reward': float(rewards.mean()),
            'std_reward': float(rewards.std()),
            'median_reward': float(np.median(rewards)),
            'min_reward': float(rewards.min()),
            'max_reward': float(rewards.max()),
            'success_rate': float(successes) / len(rewards),
            'crash_rate': float(crashes) / len(rewards),
            'mean_episode_length': float(episode_lengths.mean()),
            'std_episode_length': float(episode_lengths.std()),
            'per_episode': rewards,
            'episode_lengths': episode_lengths
        }
        
        # Logging melhorado
        print(f"\n{'='*60}")
        print(f"EVALUATION SUMMARY")
        print(f"{'='*60}")
        print(f"  Episodes: {episodes}")
        print(f"  Mean Reward: {result['mean_reward']:.2f} ± {result['std_reward']:.2f}")
        print(f"  Median Reward: {result['median_reward']:.2f}")
        print(f"  Min/Max Reward: {result['min_reward']:.2f} / {result['max_reward']:.2f}")
        print(f"  Success Rate: {result['success_rate']:.2%}")
        print(f"  Crash Rate: {result['crash_rate']:.2%}")
        print(f"  Mean Episode Length: {result['mean_episode_length']:.1f} ± {result['std_episode_length']:.1f} steps")
        print(f"{'='*60}\n")
        
        return result
        
    finally:
        # Garantir que ambiente fecha mesmo com erros
        if vec_env is not None:
            vec_env.close()

# Função para plot de treino

In [12]:
def plot_training_monitor(monitor_csv_path, window=10, show=True, out_png=None):
    """Plota rewards (raw + smoothed) a partir do monitor.csv gerado pelo Monitor."""
    if not os.path.exists(monitor_csv_path):
        print("[PLOT] monitor file not found:", monitor_csv_path)
        return
    
    fig = None
    try:
        # Carregar dados com tratamento de erros
        try:
            df = pd.read_csv(monitor_csv_path, comment='#')
        except Exception as e:
            print(f"[PLOT ERROR] Failed to read CSV: {e}")
            return
        
        if df.empty or 'r' not in df.columns:
            print("[PLOT ERROR] CSV is empty or missing 'r' column")
            return
        
        # Calcular smoothing
        df['r_smooth'] = df['r'].rolling(window=window, min_periods=1).mean()
        
        # Criar figura
        fig = plt.figure(figsize=(12, 5))
        
        # Plot raw rewards (cinza transparente)
        plt.plot(df['r'], alpha=0.2, color='gray', linewidth=0.8, label='Raw rewards')
        
        # Plot smoothed rewards (azul destacado)
        plt.plot(df['r_smooth'], color='#2E86DE', linewidth=2.5, label=f'Smoothed (window={window})')
        
        # Linha de referência em y=0
        plt.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Target (0)')
        
        # Adicionar linha de referência em y=200 (sucesso)
        plt.axhline(y=200, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Success (200)')
        
        # Labels e título
        plt.xlabel('Episode', fontsize=12, fontweight='bold')
        plt.ylabel('Reward', fontsize=12, fontweight='bold')
        
        # Título com estatísticas
        config_name = os.path.basename(os.path.dirname(monitor_csv_path))
        mean_reward = df['r'].mean()
        final_avg = df['r_smooth'].iloc[-50:].mean() if len(df) >= 50 else df['r_smooth'].mean()
        plt.title(f'{config_name} | Mean: {mean_reward:.1f} | Final 50 eps avg: {final_avg:.1f}',
                 fontsize=13, fontweight='bold', pad=15)
        
        # Grid e legend
        plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
        plt.legend(loc='lower right', fontsize=10, framealpha=0.9)
        
        # Ajustar layout
        plt.tight_layout()
        
        # Salvar se especificado
        if out_png:
            plt.savefig(out_png, dpi=150, bbox_inches='tight')
            print(f"[PLOT] ✅ Saved to {out_png}")
        
        # Mostrar se especificado
        if show:
            plt.show()
            
    except Exception as e:
        print(f"[PLOT ERROR] Unexpected error: {e}")
        
    finally:
        # Garantir fechamento da figura
        if fig is not None:
            plt.close(fig)

def plot_compare_configs(base_dir, configs, seed, window=20):
    """Plota curvas suavizadas (por seed) comparando várias configs no mesmo gráfico."""
    plt.figure(figsize=(10,5))
    for cfg in configs:
        monitor_csv = os.path.join(base_dir, cfg, f'seed{seed}', 'monitor.csv')
        if not os.path.exists(monitor_csv):
            print("[COMPARE] monitor not found for", cfg, "seed", seed)
            continue
        df = pd.read_csv(monitor_csv, comment='#')
        df['r_smooth'] = df['r'].rolling(window=window, min_periods=1).mean()
        plt.plot(df['r_smooth'], label=f'{cfg}')
    plt.xlabel('Episode')
    plt.ylabel('Smoothed Reward')
    plt.title(f'Compare configs (seed {seed})')
    plt.legend()
    plt.show()

# Execução para múltiplas seeds

In [13]:
def run_experiments(
    configs,
    seeds,
    timesteps=50_000,
    hyperparams=None,
    out_dir='./experiments',
    name='experiment'
):
    """
    Roda treinos e avaliações para lista de configs e seeds.
    Retorna: results[config][seed] = metrics
    """
    results = {}

    for config in configs:
        results[config] = {}

        for seed in seeds:
            print("\n============================")
            print(f"Training config={config} seed={seed}")

            model_path, run_dir = train_configs(
                config_name=config,
                seed=seed,
                timesteps=timesteps,
                hyperparams=hyperparams,
                out_dir=out_dir,
                name=name
            )

            print(f"Evaluating model for config={config} seed={seed}")

            metrics = evaluate_custom(
                model_path=model_path,
                config_name=config,
                seed=seed,
                episodes=50
            )

            results[config][seed] = metrics

    return results


In [14]:
from itertools import product

def run_hyperparam_grid(config_name, seed, grid, timesteps=300_000, out_root='./experiments_grid'):
    """
    grid: dict of lists, e.g. {'learning_rate':[3e-5,1e-4],'n_steps':[2048,4096]}
    Vai gerar todas as combinações, treinar e guardar resultados em out_root/config_name/seed/hparam_i
    Retorna lista de (hparam_dict, metrics)
    """
    keys, values = zip(*grid.items())
    combos = [dict(zip(keys, v)) for v in product(*values)]
    results = []
    for i, combo in enumerate(combos):
        print(f"\n---- Grid {i+1}/{len(combos)}: {combo}")
        # colocar cada combo numa subpasta
        out_root_combo = os.path.join(out_root, config_name, f'seed{seed}', f'grid_{i}')
        os.makedirs(out_root_combo, exist_ok=True)
        model_path, _ = train_configs(config_name=config_name, seed=seed, timesteps=timesteps,
                                    hyperparams=combo, out_root=out_root_combo)
        metrics = evaluate_custom(model_path=model_path, config_name=config_name, seed=seed, episodes=30)
        results.append((combo, metrics, model_path, out_root_combo))
    return results


In [15]:
def run_hyperparam_grid_optimized(config_name, seed, grid, timesteps=300_000, out_root='./experiments_grid', 
                                    min_success_rate=0.7, top_k=3):
    """
    Grid search OTIMIZADO com tracking de melhores modelos.
    
    Args:
        min_success_rate: Taxa mínima de sucesso para considerar modelo promissor
        top_k: Guardar apenas os top K melhores modelos
        
    Returns:
        best_results: Lista dos K melhores (hparam_dict, metrics, model_path)
    """
    from itertools import product
    
    keys, values = zip(*grid.items())
    combos = [dict(zip(keys, v)) for v in product(*values)]
    results = []
    best_results = []
    
    print(f"\n{'='*60}")
    print(f"HYPERPARAMETER OPTIMIZATION")
    print(f"Total combinations: {len(combos)}")
    print(f"{'='*60}\n")
    
    for i, combo in enumerate(combos):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(combos)}] Testing: {combo}")
        print(f"{'='*60}")
        
        # Treinar modelo
        out_root_combo = os.path.join(out_root, config_name, f'seed{seed}', f'grid_{i}')
        os.makedirs(out_root_combo, exist_ok=True)
        
        try:
            model_path, _ = train_configs(
                config_name=config_name, 
                seed=seed, 
                timesteps=timesteps,
                hyperparams=combo, 
                out_dir=out_root_combo,
                name=f'grid_{i}'
            )
            
            # Avaliar modelo
            metrics = evaluate_custom(
                model_path=model_path, 
                config_name=config_name, 
                seed=seed, 
                episodes=50
            )
            
            result = {
                'combo': combo,
                'metrics': metrics,
                'model_path': model_path,
                'out_dir': out_root_combo,
                'score': metrics['mean_reward'] + metrics['success_rate'] * 100  # Score combinado
            }
            results.append(result)
            
            # Print resumo
            print(f"\n📊 RESULTS:")
            print(f"  Mean Reward: {metrics['mean_reward']:.2f}")
            print(f"  Success Rate: {metrics['success_rate']:.2%}")
            print(f"  Combined Score: {result['score']:.2f}")
            
            # Atualizar top-k
            best_results.append(result)
            best_results.sort(key=lambda x: x['score'], reverse=True)
            best_results = best_results[:top_k]
            
            print(f"\n🏆 CURRENT TOP {top_k}:")
            for idx, res in enumerate(best_results, 1):
                print(f"  {idx}. Score={res['score']:.2f} | {res['combo']}")
            
        except Exception as e:
            print(f"\n❌ ERROR training combo {i+1}: {e}")
            continue
    
    # Resultado final
    print(f"\n\n{'='*60}")
    print(f"🎯 FINAL TOP {top_k} HYPERPARAMETERS")
    print(f"{'='*60}\n")
    
    for idx, res in enumerate(best_results, 1):
        print(f"\n{'='*60}")
        print(f"#{idx} - Score: {res['score']:.2f}")
        print(f"{'='*60}")
        print(f"Hyperparameters:")
        for k, v in res['combo'].items():
            print(f"  - {k}: {v}")
        print(f"\nMetrics:")
        print(f"  - Mean Reward: {res['metrics']['mean_reward']:.2f}")
        print(f"  - Success Rate: {res['metrics']['success_rate']:.2%}")
        print(f"  - Model Path: {res['model_path']}")
    
    return best_results, results

In [16]:
def run_random_search(config_name, seed, param_distributions, n_trials=20, timesteps=300_000, 
                      out_root='./experiments_random'):
    """
    Random Search: mais eficiente que Grid Search.
    
    Args:
        param_distributions: dict com ranges, ex:
            {
                'learning_rate': (1e-5, 1e-3, 'log'),  # log-uniform
                'n_steps': [1024, 2048, 4096],         # choice
                'batch_size': (32, 128, 'int'),        # uniform int
                'ent_coef': (0.0, 0.1, 'uniform')      # uniform float
            }
        n_trials: número de combinações aleatórias a testar
    """
    import random as rand
    
    def sample_param(distribution):
        """Amostra um parâmetro da distribuição"""
        if isinstance(distribution, list):
            # Choice
            return rand.choice(distribution)
        elif isinstance(distribution, tuple):
            low, high, dist_type = distribution
            if dist_type == 'log':
                return 10 ** rand.uniform(np.log10(low), np.log10(high))
            elif dist_type == 'int':
                return rand.randint(int(low), int(high))
            elif dist_type == 'uniform':
                return rand.uniform(low, high)
        return distribution
    
    results = []
    best_score = -float('inf')
    best_combo = None
    
    print(f"\n{'='*60}")
    print(f"RANDOM SEARCH OPTIMIZATION")
    print(f"Trials: {n_trials}")
    print(f"{'='*60}\n")
    
    for trial in range(n_trials):
        # Sample hyperparameters
        combo = {key: sample_param(dist) for key, dist in param_distributions.items()}
        
        print(f"\n{'='*60}")
        print(f"Trial {trial+1}/{n_trials}")
        print(f"{'='*60}")
        for k, v in combo.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.6f}")
            else:
                print(f"  {k}: {v}")
        
        # Treinar
        out_dir = os.path.join(out_root, config_name, f'seed{seed}', f'trial_{trial}')
        os.makedirs(out_dir, exist_ok=True)
        
        try:
            model_path, _ = train_configs(
                config_name=config_name,
                seed=seed,
                timesteps=timesteps,
                hyperparams=combo,
                out_dir=out_dir,
                name=f'trial_{trial}'
            )
            
            metrics = evaluate_custom(model_path, config_name, seed, episodes=50)
            score = metrics['mean_reward'] + metrics['success_rate'] * 100
            
            results.append({
                'combo': combo,
                'metrics': metrics,
                'score': score,
                'model_path': model_path
            })
            
            print(f"\n📊 Score: {score:.2f} (reward={metrics['mean_reward']:.2f}, success={metrics['success_rate']:.2%})")
            
            if score > best_score:
                best_score = score
                best_combo = combo
                print(f"🏆 NEW BEST!")
            
        except Exception as e:
            print(f"❌ ERROR: {e}")
            continue
    
    # Ordenar resultados
    results.sort(key=lambda x: x['score'], reverse=True)
    
    print(f"\n\n{'='*60}")
    print(f"🎯 BEST HYPERPARAMETERS (Score: {best_score:.2f})")
    print(f"{'='*60}")
    for k, v in best_combo.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.6f}")
        else:
            print(f"  {k}: {v}")
    
    return results

# 🎯 Pipeline Completo de Otimização de Hiperparâmetros

Pipeline em 4 fases: Random Search → Grid Search → Validação Multi-Seed → Análise Final

In [17]:
def hyperparameter_optimization_pipeline(
    config_name='reward',
    n_random_trials=30,
    timesteps_phase1=200_000,
    timesteps_phase2=300_000,
    timesteps_phase3=500_000,
    out_root='./hpo_pipeline'
):
    """
    Pipeline completo de otimização de hiperparâmetros em 4 fases:
    
    FASE 1: Random Search (20-40 trials) - Exploração ampla
    FASE 2: Grid Search - Refinamento local dos melhores
    FASE 3: Validação Multi-Seed - 3 seeds diferentes
    FASE 4: Análise e Comparação Final
    
    Args:
        config_name: Configuração do ambiente ('reward', 'wind', etc.)
        n_random_trials: Número de trials no Random Search (20-40 recomendado)
        timesteps_phase1: Timesteps para fase 1 (mais rápido)
        timesteps_phase2: Timesteps para fase 2 (médio)
        timesteps_phase3: Timesteps para fase 3 (completo)
        out_root: Diretório raiz para salvar resultados
    """
    import json
    
    print("\n" + "="*80)
    print("🎯 HYPERPARAMETER OPTIMIZATION PIPELINE")
    print("="*80)
    print(f"Config: {config_name}")
    print(f"Phase 1: Random Search with {n_random_trials} trials")
    print(f"Phase 2: Grid Search refinement")
    print(f"Phase 3: Multi-seed validation (3 seeds)")
    print("="*80 + "\n")
    
    # ========================================================================
    # FASE 1: RANDOM SEARCH - Exploração Ampla
    # ========================================================================
    print("\n" + "🔍 " + "="*76)
    print("FASE 1: RANDOM SEARCH - Exploração Ampla")
    print("="*80)
    
    # Definir ranges AMPLOS para exploração
    param_distributions = {
        'learning_rate': (5e-6, 1e-3, 'log'),      # Range muito amplo
        'n_steps': [1024, 2048, 4096, 8192],       # Várias opções
        'batch_size': [32, 64, 128, 256],          # Pequeno a grande
        'n_epochs': (5, 20, 'int'),                # 5 a 20 epochs
        'gamma': (0.95, 0.9999, 'uniform'),        # Discount factor
        'gae_lambda': (0.85, 0.99, 'uniform'),     # GAE lambda
        'ent_coef': (0.0, 0.05, 'uniform'),        # Entropy bonus
        'clip_range': (0.1, 0.4, 'uniform'),       # PPO clip
        'vf_coef': (0.3, 0.7, 'uniform')           # Value function coef
    }
    
    phase1_dir = os.path.join(out_root, 'phase1_random_search')
    
    phase1_results = run_random_search(
        config_name=config_name,
        seed=42,  # Seed fixo para fase 1
        param_distributions=param_distributions,
        n_trials=n_random_trials,
        timesteps=timesteps_phase1,
        out_root=phase1_dir
    )
    
    # Selecionar TOP 3 melhores
    top3_phase1 = phase1_results[:3]
    
    print("\n✅ FASE 1 CONCLUÍDA")
    print(f"Top 3 combinações:")
    for idx, res in enumerate(top3_phase1, 1):
        print(f"  {idx}. Score: {res['score']:.2f} | Reward: {res['metrics']['mean_reward']:.2f} | Success: {res['metrics']['success_rate']:.2%}")
    
    # ========================================================================
    # FASE 2: GRID SEARCH - Refinamento Local
    # ========================================================================
    print("\n" + "🔬 " + "="*76)
    print("FASE 2: GRID SEARCH - Refinamento Local")
    print("="*80)
    
    # Construir grid refinado baseado no melhor da Fase 1
    best_phase1 = top3_phase1[0]['combo']
    
    # Criar variações ao redor dos melhores valores
    def create_refinement_grid(best_params):
        """Cria grid de refinamento ao redor dos melhores parâmetros"""
        grid = {}
        
        # Learning rate: ±30% do melhor
        lr = best_params['learning_rate']
        grid['learning_rate'] = [lr * 0.7, lr, lr * 1.3]
        
        # N_steps: valor melhor ± 1 nível
        n_steps = best_params['n_steps']
        steps_options = [1024, 2048, 4096, 8192]
        idx = steps_options.index(n_steps)
        grid['n_steps'] = [steps_options[max(0, idx-1)], n_steps, steps_options[min(len(steps_options)-1, idx+1)]]
        grid['n_steps'] = list(set(grid['n_steps']))  # Remove duplicados
        
        # Batch size: melhor ± 1 nível
        batch = best_params['batch_size']
        batch_options = [32, 64, 128, 256]
        idx = batch_options.index(batch)
        grid['batch_size'] = [batch_options[max(0, idx-1)], batch, batch_options[min(len(batch_options)-1, idx+1)]]
        grid['batch_size'] = list(set(grid['batch_size']))
        
        # Gamma: ±0.01
        gamma = best_params['gamma']
        grid['gamma'] = [max(0.95, gamma - 0.01), gamma, min(0.9999, gamma + 0.01)]
        
        # Ent_coef: ±30%
        ent = best_params['ent_coef']
        grid['ent_coef'] = [max(0.0, ent * 0.7), ent, min(0.05, ent * 1.3)]
        
        return grid
    
    refinement_grid = create_refinement_grid(best_phase1)
    
    print(f"\nGrid de refinamento criado:")
    for key, values in refinement_grid.items():
        print(f"  {key}: {values}")
    
    phase2_dir = os.path.join(out_root, 'phase2_grid_refinement')
    
    best_phase2, all_phase2 = run_hyperparam_grid_optimized(
        config_name=config_name,
        seed=42,
        grid=refinement_grid,
        timesteps=timesteps_phase2,
        out_root=phase2_dir,
        top_k=3
    )
    
    print("\n✅ FASE 2 CONCLUÍDA")
    print(f"Melhor combinação refinada:")
    print(f"  Score: {best_phase2[0]['score']:.2f}")
    print(f"  Reward: {best_phase2[0]['metrics']['mean_reward']:.2f}")
    print(f"  Success: {best_phase2[0]['metrics']['success_rate']:.2%}")
    
    # ========================================================================
    # FASE 3: VALIDAÇÃO MULTI-SEED
    # ========================================================================
    print("\n" + "🧪 " + "="*76)
    print("FASE 3: VALIDAÇÃO MULTI-SEED - Testar Robustez")
    print("="*80)
    
    best_hyperparams = best_phase2[0]['combo']
    validation_seeds = [42, 123, 777]
    
    phase3_results = []
    
    for seed_idx, seed in enumerate(validation_seeds, 1):
        print(f"\n--- Validação {seed_idx}/3 (seed={seed}) ---")
        
        phase3_dir = os.path.join(out_root, 'phase3_validation', f'seed_{seed}')
        os.makedirs(phase3_dir, exist_ok=True)
        
        model_path, _ = train_configs(
            config_name=config_name,
            seed=seed,
            timesteps=timesteps_phase3,
            hyperparams=best_hyperparams,
            out_dir=phase3_dir,
            name=f'final_seed{seed}'
        )
        
        metrics = evaluate_custom(
            model_path=model_path,
            config_name=config_name,
            seed=seed,
            episodes=50 
        )
        
        phase3_results.append({
            'seed': seed,
            'metrics': metrics,
            'model_path': model_path
        })
        
        print(f"✅ Seed {seed}: Reward={metrics['mean_reward']:.2f}, Success={metrics['success_rate']:.2%}")
    
    # ========================================================================
    # FASE 4: ANÁLISE FINAL E COMPARAÇÃO
    # ========================================================================
    print("\n" + "📊 " + "="*76)
    print("FASE 4: ANÁLISE FINAL - Resultados Consolidados")
    print("="*80)
    
    # Calcular estatísticas agregadas
    rewards = [r['metrics']['mean_reward'] for r in phase3_results]
    success_rates = [r['metrics']['success_rate'] for r in phase3_results]
    
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    mean_success = np.mean(success_rates)
    std_success = np.std(success_rates)
    
    print("\n" + "="*80)
    print("🏆 MELHORES HIPERPARÂMETROS ENCONTRADOS")
    print("="*80)
    for key, value in best_hyperparams.items():
        if isinstance(value, float):
            print(f"  {key:20s}: {value:.6f}")
        else:
            print(f"  {key:20s}: {value}")
    
    print("\n" + "="*80)
    print("📈 PERFORMANCE FINAL (3 seeds)")
    print("="*80)
    print(f"  Mean Reward:    {mean_reward:.2f} ± {std_reward:.2f}")
    print(f"  Success Rate:   {mean_success:.2%} ± {std_success:.2%}")
    print(f"\n  Resultados por seed:")
    for res in phase3_results:
        print(f"    Seed {res['seed']:3d}: Reward={res['metrics']['mean_reward']:7.2f}, Success={res['metrics']['success_rate']:.2%}, "
              f"Crash={res['metrics']['crash_rate']:.2%}")
    
    print("\n" + "="*80)
    print("📁 RESUMO DE FASES")
    print("="*80)
    print(f"  Fase 1: {n_random_trials} trials explorados")
    print(f"  Fase 2: {len(all_phase2)} combinações refinadas")
    print(f"  Fase 3: 3 seeds validados")
    print(f"  Melhor modelo: {phase3_results[np.argmax(rewards)]['model_path']}")
    
    # Salvar resultados em JSON
    summary = {
        'best_hyperparams': best_hyperparams,
        'phase1_top3': [{'score': r['score'], 'combo': r['combo']} for r in top3_phase1],
        'phase2_best': {'score': best_phase2[0]['score'], 'combo': best_phase2[0]['combo']},
        'phase3_validation': {
            'mean_reward': float(mean_reward),
            'std_reward': float(std_reward),
            'mean_success_rate': float(mean_success),
            'std_success_rate': float(std_success),
            'seeds': [{'seed': r['seed'], 'reward': r['metrics']['mean_reward'], 
                      'success_rate': r['metrics']['success_rate']} for r in phase3_results]
        }
    }
    
    summary_path = os.path.join(out_root, 'optimization_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n💾 Resumo salvo em: {summary_path}")
    print("\n" + "="*80)
    print("✅ PIPELINE DE OTIMIZAÇÃO CONCLUÍDO COM SUCESSO!")
    print("="*80 + "\n")
    
    return {
        'best_hyperparams': best_hyperparams,
        'phase1_results': phase1_results,
        'phase2_results': all_phase2,
        'phase3_validation': phase3_results,
        'summary': summary
    }

## Pipeline do Hyper-Parametrization-Optimizer

In [ ]:
# FASE 1: Executar pipeline de otimização de hyperparameters (2-4 horas)
print("="*70)
print("INICIANDO OTIMIZAÇÃO DE HYPERPARAMETERS")
print("="*70)
print(f"Config: {name}")
print(f"Tempo estimado: 2-4 horas (versão rápida)")
print("="*70 + "\n")

hpo_results = hyperparameter_optimization_pipeline(
    config_name=name,
    n_random_trials=15,              # Fast version
    timesteps_phase1=100_000,        # Phase 1: Random Search
    timesteps_phase2=150_000,        # Phase 2: Grid Refinement
    timesteps_phase3=250_000,        # Phase 3: Multi-seed validation
    out_root=f'./hpo_pipeline_{name}'
)

# Extrair melhores hyperparameters
best_hp = hpo_results['best_hyperparams']

print("\n" + "="*70)
print("MELHORES HYPERPARAMETERS ENCONTRADOS:")
print("="*70)
for key, value in best_hp.items():
    print(f"  {key:20s} = {value}")
print("="*70 + "\n")


🎯 VERSÃO COMPLETA (comentada):

⚡ VERSÃO RÁPIDA:
✅ Pipeline de otimização pronto!

📝 Para executar, descomenta o código acima


# Test Models

In [19]:
def visualize_model(model_path, config_name='orig', episodes=5, seed=None, render_mode='human'):
    """
    Carrega um modelo treinado e renderiza episódios para visualização.
    
    Args:
        model_path: caminho para o ficheiro .zip do modelo
        config_name: configuração do ambiente ('orig', 'reward', 'wind', 'noise', 'smooth', 'all')
        episodes: número de episódios para visualizar
        seed: seed para reprodutibilidade (aplicada a cada reset)
        render_mode: 'human' para janela ou 'rgb_array' para gravar vídeo
        
    Returns:
        dict: {'rewards': list, 'mean_reward': float, 'episode_lengths': list}
    """
    env = None
    try:
        # Criar ambiente COM renderização
        env = gym.make('LunarLander-v3', render_mode=render_mode, max_episode_steps=1200)
        
        # Aplicar mesmos wrappers usados no treino (ordem correta!)
        if config_name == 'all':
            print("[VISUALIZE] Applying all wrappers")
            env = RewardShapingWrapper(env)
            env = WindForceWrapper(env, wind_strength=0.03, deterministic=False)
            env = ObservationNoiseWrapper(env, noise_std=0.02)
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        elif config_name == 'reward':
            print("[VISUALIZE] Applying RewardShapingWrapper")
            env = RewardShapingWrapper(env)
        elif config_name == 'wind':
            print("[VISUALIZE] Applying WindForceWrapper")
            env = WindForceWrapper(env, wind_strength=0.01, deterministic=False)
        elif config_name == 'noise':
            print("[VISUALIZE] Applying ObservationNoiseWrapper")
            env = ObservationNoiseWrapper(env, noise_std=0.02)
        elif config_name == 'smooth':
            print("[VISUALIZE] Applying ActionSmoothingWrapper")
            env = ActionSmoothingWrapper(env, smoothing_factor=0.3)
        else:
            print("[VISUALIZE] No wrappers (original environment)")
        
        # Carregar modelo
        print(f"[VISUALIZE] Loading model from: {model_path}")
        model = PPO.load(model_path)
        
        # Executar episódios
        rewards_list = []
        lengths_list = []
        
        for ep in range(episodes):
            # Reset com seed específica para cada episódio (se fornecida)
            if seed is not None:
                obs, _ = env.reset(seed=seed + ep)
            else:
                obs, _ = env.reset()
            
            done = False
            total_reward = 0
            steps = 0
            
            print(f"\n{'='*60}")
            print(f"  Episódio {ep+1}/{episodes}")
            print(f"{'='*60}")
            
            while not done:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                total_reward += reward
                steps += 1
                
                # Não precisa de env.render() com render_mode='human'
                # O Gymnasium já renderiza automaticamente
                
                if done and render_mode == 'human':
                    time.sleep(1.0)  # Pausa no final para ver resultado
            
            rewards_list.append(total_reward)
            lengths_list.append(steps)
            
            # Status do episódio
            status = "✓ SUCCESS" if total_reward >= 200 else "✗ FAIL"
            print(f"  Reward: {total_reward:7.2f} | Steps: {steps:4d} | {status}")
        
        # Resumo final
        mean_reward = np.mean(rewards_list)
        std_reward = np.std(rewards_list)
        success_rate = sum(1 for r in rewards_list if r >= 200) / len(rewards_list)
        
        print(f"\n{'='*60}")
        print(f"RESUMO DA VISUALIZAÇÃO")
        print(f"{'='*60}")
        print(f"  Episódios:      {episodes}")
        print(f"  Reward médio:   {mean_reward:.2f} ± {std_reward:.2f}")
        print(f"  Taxa sucesso:   {success_rate*100:.1f}%")
        print(f"  Steps médio:    {np.mean(lengths_list):.1f}")
        print(f"{'='*60}\n")
        
        return {
            'rewards': rewards_list,
            'mean_reward': mean_reward,
            'std_reward': std_reward,
            'success_rate': success_rate,
            'episode_lengths': lengths_list
        }
        
    finally:
        if env is not None:
            env.close()
            print("[VISUALIZE] Environment closed.")

In [20]:
configs = ['reward']
seeds = [random.randint(0, 100)]
out_root = './experiments' 
name = ''

for i, c in enumerate(configs):
    name += c
    if (i < len(configs) - 1):
        name += '_'


out_dir = os.path.join(out_root, name, f'seed{seeds[0]}')
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# FASE 1: Executar pipeline de otimização de hyperparameters (2-4 horas)
print("="*70)
print("INICIANDO OTIMIZAÇÃO DE HYPERPARAMETERS")
print("="*70)
print(f"Config: {name}")
print(f"Tempo estimado: 2-4 horas (versão rápida)")
print("="*70 + "\n")

hpo_results = hyperparameter_optimization_pipeline(
    config_name=name,
    n_random_trials=15,              # Fast version
    timesteps_phase1=100_000,        # Phase 1: Random Search
    timesteps_phase2=150_000,        # Phase 2: Grid Refinement
    timesteps_phase3=250_000,        # Phase 3: Multi-seed validation
    out_root=f'./hpo_pipeline_{name}'
)

# Extrair melhores hyperparameters
best_hp = hpo_results['best_hyperparams']

print("\n" + "="*70)
print("MELHORES HYPERPARAMETERS ENCONTRADOS:")
print("="*70)
for key, value in best_hp.items():
    print(f"  {key:20s} = {value}")
print("="*70 + "\n")

In [ ]:
# FASE 2: Treinar modelo final com hyperparameters otimizados
print("="*70)
print("TREINANDO MODELO FINAL COM HYPERPARAMETERS OTIMIZADOS")
print("="*70)
print(f"Timesteps: 500,000")
print(f"Config: {name}")
print(f"Seeds: {seeds}")
print("="*70 + "\n")

results = run_experiments(
    configs=configs,
    seeds=seeds,
    timesteps=500_000,
    hyperparams=best_hp,  # ✓ Usa hyperparameters otimizados do pipeline!
    out_dir=out_dir,
    name=name
)

print("\n" + "="*70)
print("TREINO COMPLETO!")
print("="*70)

<VSCode.Cell id="#VSC-92c5109e" language="python">
# Visualizar o modelo treinado
model_path = f'{out_dir}/ppo_{name}_seed{seeds[0]}'

visualize_model(
    model_path=model_path,
    config_name=name,  # Usa 'reward' (string), não lista
    episodes=10,
    seed=42  # Seed fixa para reprodutibilidade
)

In [ ]:
tensorboard_log=os.path.join(out_dir, 'tb')

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
import glob

def load_tensorboard_logs(log_dir):
    """
    Carrega dados do TensorBoard a partir da pasta de logs.
    Retorna dict com todas as métricas: loss, reward, etc.
    """
    # Encontrar ficheiros de eventos (tfevents)
    event_files = glob.glob(os.path.join(log_dir, '**/events.out.tfevents.*'), recursive=True)
    
    if not event_files:
        print(f"[ERROR] Nenhum ficheiro TensorBoard encontrado em {log_dir}")
        return None
    
    print(f"[INFO] Encontrados {len(event_files)} ficheiros de eventos")
    
    # Carregar eventos
    ea = event_accumulator.EventAccumulator(event_files[0])
    ea.Reload()
    
    # Extrair todas as métricas disponíveis
    data = {}
    
    # Listar tags disponíveis
    scalar_tags = ea.Tags()['scalars']
    print(f"[INFO] Métricas disponíveis: {scalar_tags}")
    
    for tag in scalar_tags:
        events = ea.Scalars(tag)
        data[tag] = {
            'steps': [e.step for e in events],
            'values': [e.value for e in events],
            'wall_time': [e.wall_time for e in events]
        }
    
    return data

def plot_tensorboard_metrics(log_dir, save_path=None):
    """
    Plota todas as métricas importantes do TensorBoard.
    """
    data = load_tensorboard_logs(log_dir)
    
    if data is None:
        return
    
    # Filtrar métricas de treino
    train_metrics = {k: v for k, v in data.items() if 'train' in k}
    rollout_metrics = {k: v for k, v in data.items() if 'rollout' in k}
    eval_metrics = {k: v for k, v in data.items() if 'eval' in k}
    
    # Criar figura com subplots
    n_train = len(train_metrics)
    n_rollout = len(rollout_metrics)
    n_eval = len(eval_metrics)
    
    total_plots = n_train + n_rollout + n_eval
    
    if total_plots == 0:
        print("[ERROR] Nenhuma métrica encontrada!")
        return
    
    # Calcular grid
    ncols = 3
    nrows = (total_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5*nrows))
    axes = axes.flatten() if total_plots > 1 else [axes]
    
    plot_idx = 0
    
    # Plot 1: Training Losses
    for metric_name, metric_data in train_metrics.items():
        ax = axes[plot_idx]
        ax.plot(metric_data['steps'], metric_data['values'], linewidth=2, alpha=0.8)
        ax.set_xlabel('Steps', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.set_title(metric_name.replace('train/', ''), fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    # Plot 2: Rollout Metrics (ep_rew_mean, ep_len_mean)
    for metric_name, metric_data in rollout_metrics.items():
        ax = axes[plot_idx]
        ax.plot(metric_data['steps'], metric_data['values'], 
                linewidth=2, alpha=0.8, color='green')
        ax.set_xlabel('Steps', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.set_title(metric_name.replace('rollout/', ''), fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    # Plot 3: Evaluation Metrics
    for metric_name, metric_data in eval_metrics.items():
        ax = axes[plot_idx]
        ax.plot(metric_data['steps'], metric_data['values'], 
                linewidth=2, alpha=0.8, color='orange')
        ax.set_xlabel('Steps', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.set_title(metric_name.replace('eval/', ''), fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    # Esconder axes vazios
    for idx in range(plot_idx, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('TensorBoard Metrics', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"[SAVE] Plot guardado em {save_path}")
    
    plt.show()

def plot_loss_only(log_dir, save_path=None):
    """
    Plota apenas as losses (policy, value, entropy) do treino.
    """
    data = load_tensorboard_logs(log_dir)
    
    if data is None:
        return
    
    # Métricas de loss
    loss_keys = ['train/policy_loss', 'train/value_loss', 'train/entropy_loss', 'train/loss']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    colors = ['blue', 'red', 'green', 'purple']
    
    for idx, (loss_key, color) in enumerate(zip(loss_keys, colors)):
        if loss_key in data:
            metric = data[loss_key]
            axes[idx].plot(metric['steps'], metric['values'], 
                          color=color, linewidth=2, alpha=0.7)
            axes[idx].set_xlabel('Training Steps', fontsize=11)
            axes[idx].set_ylabel('Loss', fontsize=11)
            axes[idx].set_title(loss_key.replace('train/', '').replace('_', ' ').title(), 
                               fontsize=13, fontweight='bold')
            axes[idx].grid(True, alpha=0.3)
        else:
            axes[idx].text(0.5, 0.5, f'{loss_key}\nnot available', 
                          ha='center', va='center', fontsize=12)
            axes[idx].axis('off')
    
    plt.suptitle('Training Losses', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"[SAVE] Loss plot guardado em {save_path}")
    
    plt.show()

In [ ]:
tensorboard_dir = os.path.join(out_dir, 'tb')
plot_tensorboard_metrics(tensorboard_dir, save_path=f'{out_dir}/all_metrics.png')

# Plot APENAS das losses
plot_loss_only(tensorboard_dir, save_path=f'{out_dir}/losses.png')

[ERROR] Nenhum ficheiro TensorBoard encontrado em ./experiments\reward\seed69\tb
[ERROR] Nenhum ficheiro TensorBoard encontrado em ./experiments\reward\seed69\tb
